# 01 — Data Collection

Two source options — run whichever is available:

| Option | Source | Requires |
|--------|--------|----------|
| **A (recommended)** | Kaggle — Sephora Reviews dataset | `kaggle` CLI + free account |
| **B (fallback)** | Reddit `.json` endpoint | Nothing — no API key |

Both produce the same output schema so the rest of the pipeline is unchanged.

In [ ]:
import os
import time
import json
import requests
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

RAW_DIR = Path('../data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

---
## Option A — Kaggle: Sephora Products & Skincare Reviews

**Dataset:** `nadyinky/sephora-products-and-skincare-reviews`  
Contains ~8,000 products and 1M+ reviews with `skin_type`, `skin_tone`, `rating`, `review_text`.

**Setup (one-time, 2 min):**
1. Create free account at kaggle.com
2. Profile → Settings → API → *Create New Token* → downloads `kaggle.json`
3. Place it at `~/.kaggle/kaggle.json` (Mac/Linux) or `C:\Users\<you>\.kaggle\kaggle.json` (Windows)
4. `pip install kaggle`

In [ ]:
# ── OPTION A: Kaggle download ─────────────────────────────────────────────
import subprocess

DATASET = 'nadyinky/sephora-products-and-skincare-reviews'
OUT_DIR = str(RAW_DIR)

result = subprocess.run(
    ['kaggle', 'datasets', 'download', '-d', DATASET, '-p', OUT_DIR, '--unzip'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)
else:
    print('Download complete ✓')
    print(list(RAW_DIR.glob('*.csv')))

In [ ]:
# Load Sephora reviews and normalise to shared schema
# The dataset has multiple review CSV files — merge them
review_files = list(RAW_DIR.glob('reviews_*.csv'))
print(f'Found {len(review_files)} review files')

dfs = [pd.read_csv(f) for f in review_files]
reviews_raw = pd.concat(dfs, ignore_index=True)
print(f'Total rows: {len(reviews_raw):,}')
reviews_raw.head(3)

In [ ]:
# Skin concern keyword mapping (same as Reddit version)
CONCERN_KEYWORDS = {
    'acne':             ['acne', 'breakout', 'pimple', 'comedone', 'cystic', 'blemish'],
    'aging':            ['wrinkle', 'fine line', 'anti-aging', 'aging', 'firm', 'collagen'],
    'sensitivity':      ['sensitive', 'redness', 'irritat', 'rosacea', 'eczema', 'reactive'],
    'hyperpigmentation':['dark spot', 'hyperpigmentation', 'melasma', 'uneven', 'brightening'],
}

def classify_concern(text: str) -> str:
    if not isinstance(text, str):
        return 'other'
    text_lower = text.lower()
    for concern, keywords in CONCERN_KEYWORDS.items():
        if any(kw in text_lower for kw in keywords):
            return concern
    return 'other'

# Map to shared schema: post_id, title, body, score, created_at, skin_concern
df_kaggle = pd.DataFrame({
    'post_id':      reviews_raw['review_id'].astype(str),
    'subreddit':    'sephora',
    'title':        reviews_raw.get('product_name', pd.Series([''] * len(reviews_raw))),
    'body':         reviews_raw['review_text'].fillna(''),
    'score':        reviews_raw['rating'].fillna(0).astype(int),
    'num_comments': 0,
    'created_at':   pd.to_datetime(reviews_raw['submission_time'], errors='coerce'),
    'skin_concern': reviews_raw['review_text'].apply(classify_concern),
})

# Filter out empty reviews
df_kaggle = df_kaggle[df_kaggle['body'].str.len() > 30].drop_duplicates('post_id')
print(f'Clean rows: {len(df_kaggle):,}')
print(df_kaggle['skin_concern'].value_counts())

In [ ]:
# Save + load to DB
df_kaggle.to_json(RAW_DIR / 'posts_raw.json', orient='records', indent=2)

DB_URL = (
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
engine = create_engine(DB_URL)
df_kaggle.to_sql('posts', engine, if_exists='append', index=False)
print(f'Loaded {len(df_kaggle):,} rows to PostgreSQL ✓')

---
## Option B — Reddit JSON (no API key needed)

Reddit exposes public `.json` endpoints without authentication.  
Rate limit: **~1 req/sec** is safe. This cell collects ~1,000 posts before the rest of the pipeline.

In [ ]:
# ── OPTION B: Reddit .json scrape ─────────────────────────────────────────
from datetime import datetime
from tqdm import tqdm

SUBREDDIT  = 'SkincareAddiction'
TARGET     = 1000   # posts to collect
SLEEP_SEC  = 1.1    # stay well under rate limit
HEADERS    = {'User-Agent': 'skincare-sentiment-analysis/1.0 (research project)'}

BASE_URL   = f'https://www.reddit.com/r/{SUBREDDIT}/new.json'

CONCERN_KEYWORDS = {
    'acne':             ['acne', 'breakout', 'pimple', 'comedone', 'cystic', 'blemish'],
    'aging':            ['wrinkle', 'fine line', 'anti-aging', 'aging', 'firm', 'collagen'],
    'sensitivity':      ['sensitive', 'redness', 'irritat', 'rosacea', 'eczema', 'reactive'],
    'hyperpigmentation':['dark spot', 'hyperpigmentation', 'melasma', 'uneven', 'brightening'],
}

def classify_concern(text: str) -> str:
    if not isinstance(text, str):
        return 'other'
    t = text.lower()
    for concern, kws in CONCERN_KEYWORDS.items():
        if any(k in t for k in kws):
            return concern
    return 'other'

records = []
after   = None

with tqdm(total=TARGET, desc='Scraping Reddit JSON') as pbar:
    while len(records) < TARGET:
        params = {'limit': 100, 'raw_json': 1}
        if after:
            params['after'] = after

        resp = requests.get(BASE_URL, headers=HEADERS, params=params, timeout=15)

        if resp.status_code == 429:
            print('Rate limited — sleeping 60s')
            time.sleep(60)
            continue
        if resp.status_code != 200:
            print(f'Error {resp.status_code} — stopping')
            break

        data     = resp.json()['data']
        children = data['children']
        if not children:
            break

        for child in children:
            post = child['data']
            body = post.get('selftext', '')
            if body in ('[deleted]', '[removed]', '') or len(body) < 50:
                continue
            full_text = post.get('title', '') + ' ' + body
            records.append({
                'post_id':      post['id'],
                'subreddit':    SUBREDDIT,
                'title':        post.get('title', ''),
                'body':         body,
                'score':        post.get('score', 0),
                'num_comments': post.get('num_comments', 0),
                'created_at':   datetime.utcfromtimestamp(post['created_utc']),
                'skin_concern': classify_concern(full_text),
            })
            pbar.update(1)

        after = data.get('after')
        if not after:
            break
        time.sleep(SLEEP_SEC)

df_reddit = pd.DataFrame(records).drop_duplicates('post_id')
print(f'\nCollected {len(df_reddit)} posts')
print(df_reddit['skin_concern'].value_counts())

In [ ]:
# Save + load to DB (same as Option A)
df_reddit.to_json(RAW_DIR / 'posts_raw.json', orient='records', indent=2)

DB_URL = (
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
engine = create_engine(DB_URL)
df_reddit.to_sql('posts', engine, if_exists='append', index=False)
print(f'Loaded {len(df_reddit)} rows to PostgreSQL ✓')